# WTI Hourly Price Data — Yahoo Finance

This notebook downloads two years of hourly OHLCV (Open, High, Low, Close, Volume) 
data for WTI Crude Oil futures (ticker: CL=F) from Yahoo Finance using the yfinance 
library. It then computes four derived liquidity metrics — log volume, price range, 
log return, and the Amihud illiquidity ratio — which serve as the primary dependent 
variables in the thesis analysis. The resulting dataset is saved to 
`01_data/raw/price/wti_hourly_raw.csv`.

### General imports

In [9]:
import yfinance as yf
import pandas as pd
import numpy as np

### Obtain prices from Yahoo Finance API

- **Open:** Price at th emoment of opening the time window.

- **High:** Highest price in the time window.

- **Low:** Lowest price in the time window.

- **Volume:** Amount of contracts on the time window.

- **Close:** Price at the moment of closing the time window.

In [10]:
df_price = yf.download("CL=F", period="2y", interval="1h", progress=False)

# If Yahoo Finance changes the order of the columns in the future, this avoid silent erros of calculating values with different columns
if isinstance(df_price.columns, pd.MultiIndex):
    df_price.columns = df_price.columns.get_level_values(0)
df_price.index = pd.to_datetime(df_price.index).tz_convert('UTC')
df_price.index.name = 'Datetime'

print(f"Registros: {len(df_price)}")
print(f"Rango: {df_price.index.min()} a {df_price.index.max()}")
print(df_price.tail(5))

Registros: 11219
Rango: 2024-03-11 10:00:00+00:00 a 2026-03-11 10:00:00+00:00
Price                          Close       High        Low       Open  Volume
Datetime                                                                     
2026-03-11 06:00:00+00:00  83.570000  84.559998  83.120003  84.010002    7734
2026-03-11 07:00:00+00:00  85.489998  85.790001  83.339996  83.570000   18061
2026-03-11 08:00:00+00:00  86.529999  86.830002  84.910004  85.449997   21511
2026-03-11 09:00:00+00:00  88.760002  88.989998  86.309998  86.529999   27162
2026-03-11 10:00:00+00:00  85.470001  88.750000  84.680000  88.750000   21920


### Calculate liquidity variables derived from the OHLCV entries

Four liquidity metrics are calculated from the raw OHLCV data:

- **log_volume:** Natural logarithm of hourly trading volume. Log transformation normalizes the right-skewed distribution of volume, making it more suitable for linear modeling.

- **price_range:** Difference between High and Low price within each hour. This serves as an intraday volatility proxy following the Parkinson (1980) extreme value estimator.

- **log_return:** Logarithmic return between consecutive hours, computed as log(Close_t / Close_{t-1}). Log returns are standard in financial analysis due to their additive properties and approximate normality.

- **amihud:** Ratio of absolute log return to trading volume, following Amihud (2002). This is the most widely cited illiquidity measure in market microstructure literature, capturing the price impact per unit of volume traded.

In [11]:
df_price['log_volume'] = np.log(df_price['Volume'] + 1)
df_price['price_range'] = df_price['High'] - df_price['Low']
df_price['log_return'] = np.log(df_price['Close'] / df_price['Close'].shift(1))
df_price['amihud'] = df_price['log_return'].abs() / (df_price['Volume'] + 1)

print(df_price[['log_volume', 'price_range', 'amihud']].describe().round(4))

Price  log_volume  price_range      amihud
count  11219.0000   11219.0000  11218.0000
mean       8.2611       0.3922      0.0001
std        2.1947       0.4419      0.0007
min        0.0000       0.0000      0.0000
25%        7.4304       0.1800      0.0000
50%        8.6663       0.3000      0.0000
75%        9.7250       0.4800      0.0000
max       13.3046      14.3900      0.0323


### Save data

In [12]:
filename = "wti_hourly_raw.csv"
df_price.to_csv(f"../01_data/raw/price/{filename}")
print(f"Guardado en 01_data/raw/price/{filename}")

Guardado en 01_data/raw/price/wti_hourly_raw.csv
